importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

importando a base de dados

In [ ]:
df = pd.read_csv("/content/FakeTrueBr_corpus.csv")

In [ ]:
df.shape[0]

1791

In [ ]:
df.head()

,title_fake,fake,link_f,true,link_t
0,vídeo mostra arrastão em olinda pe por causa d...,carnaval em olinda. arrastão monstro. fazuele ...,https://www.boatos.org/entretenimento/video-ar...,circula pelas redes sociais um vídeo que most...,https://g1.globo.com/pe/pernambuco/carnaval/20...
1,carro alegórico da grande rio em homenagem ao ...,carro alegórico da escola de samba grande rio...,https://www.boatos.org/entretenimento/carro-al...,pouco antes de a beijaflor de nilópolis entra...,https://g1.globo.com/rj/rio-de-janeiro/carnava...
2,cantor leo chaves grava vídeo de apoio a manif...,cantor léo apoiando atos antidemocráticos. alg...,https://www.boatos.org/entretenimento/cantor-l...,circula nas redes sociais um vídeo que mostra...,https://g1.globo.com/fato-ou-fake/noticia/2023...
3,anitta está com câncer e pede por um milagre e...,versão 1 ore muito pela cantora anitta! ela es...,https://www.boatos.org/entretenimento/anitta-e...,"anitta explicou, nesta terçafeira 7, o proble...",https://g1.globo.com/pop-arte/noticia/2022/06/...
4,erasmo carlos morre aos 81 anos no rio de jane...,morre erasmo carlos aos 81 anos cantor conside...,https://www.boatos.org/entretenimento/erasmo-c...,"o cantor e compositor erasmo carlos, de 81 an...",https://g1.globo.com/rj/rio-de-janeiro/noticia...


verificando nulos

In [ ]:
df.isna().sum()

,0
title_fake,0
fake,0
link_f,0
true,0
link_t,0


**separar as colunas de fake news e true em 2 linhas diferentes**


*   1 = Fake
*   0 = True



separar as noticias falsas

In [ ]:
#pega somente os registros fake
fakes = df[['fake']].rename(columns = {'fake': 'texto'})

In [ ]:
#add o valor 1, sendo 1 = notícia falsa
fakes['label'] = 1

separar as noticias verdadeiras

In [ ]:
#pega somente os registros verdadeiros
trues = df[['true']].rename(columns = {'true': 'texto'})

In [ ]:
#add o valor 0, sendo 0 = notícia verdadeira
trues['label'] = 0

transformando tudo em 1 df só

In [ ]:
#juntando os 2 df em 1 só. 1 embaixo do outro
df_full = pd.concat([fakes, trues], ignore_index= True)

In [ ]:
df_full

,texto,label
0,carnaval em olinda. arrastão monstro. fazuele ...,1
1,carro alegórico da escola de samba grande rio...,1
2,cantor léo apoiando atos antidemocráticos. alg...,1
3,versão 1 ore muito pela cantora anitta! ela es...,1
4,morre erasmo carlos aos 81 anos cantor conside...,1
...,...,...
3577,O preço médio da gasolina nos postos do Brasi...,0
3578,Não há nenhuma comprovação que vacinas contra...,0
3579,A seringa que foi usada por uma técnica de en...,0
3580,A SESAI (Secretaria Especial de Saúde Indígen...,0


In [ ]:
df_full.shape[0]

3582

**PRE-PROCESSAMENTO**

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

sanitizar o texto - retirar, por ex: 'o', 'de', 'com', pontuações

In [ ]:
#baixar stopwords em português
nltk.download('stopwords')
stop_words = stopwords.words('portuguese')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Função de limpeza de texto
def limpar_texto(texto):
    texto = str(texto).lower() # Tudo em minúsculas
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE) # Remove URLs
    texto = re.sub(r'[^\w\s]', '', texto) # Remove pontuações/caracteres especiais
    texto = re.sub(r'\d+', '', texto) # Remove números
    palavras = texto.split()
    palavras_filtradas = [w for w in palavras if w not in stop_words] # Remove stopwords
    return " ".join(palavras_filtradas)

limpar os textos das notícias

In [ ]:
df_full['texto_limpo'] = df_full['texto'].apply(limpar_texto)

In [ ]:
df_full

,texto,label,texto_limpo
0,carnaval em olinda. arrastão monstro. fazuele ...,1,carnaval olinda arrastão monstro fazuele aperi...
1,carro alegórico da escola de samba grande rio...,1,carro alegórico escola samba grande rio homena...
2,cantor léo apoiando atos antidemocráticos. alg...,1,cantor léo apoiando atos antidemocráticos algo...
3,versão 1 ore muito pela cantora anitta! ela es...,1,versão ore cantora anitta doente somente milag...
4,morre erasmo carlos aos 81 anos cantor conside...,1,morre erasmo carlos anos cantor considerado ma...
...,...,...,...
3577,O preço médio da gasolina nos postos do Brasi...,0,preço médio gasolina postos brasil ultrapassou...
3578,Não há nenhuma comprovação que vacinas contra...,0,nenhuma comprovação vacinas contra covid possa...
3579,A seringa que foi usada por uma técnica de en...,0,seringa usada técnica enfermagem simulou vacin...
3580,A SESAI (Secretaria Especial de Saúde Indígen...,0,sesai secretaria especial saúde indígena ligad...


dividindo df em teste e treino

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_full['texto_limpo'],
    df_full['label'],
    test_size = 0.2, # 20% de pra teste
    random_state = 123, #sementre da amostra
    stratify = df_full['label']
)

1. usa-se 'raw' nos X porque eles ainda são testos puros, ainda não viraram matrizes numéricas

2. estamos usando uma amostragem estratificada para garantir que a proporção de fakes e trues seja a mesma nas bases de treino e teste. Isso porque uma AAS pode, ao azar, escolher mais de 1 classe do que de outra.

Vetorização TF-IDF

In [ ]:
#transformar os dados em matrizes numéricas
vectorizer = TfidfVectorizer(max_features=5000) # mantém as 5k palavras mais importantes

X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

# **Treinando Naive Bayes**



*   X_train_tfidf = dados de treino vetorizados
*   y_train = gabaarito dos dados de treino



In [ ]:
#instanciar o modelo NB
nb_modelo = MultinomialNB()

#treinando o modelo
nb_modelo.fit(X_train_tfidf, y_train)


MultinomialNB()

Teste: com o modelo já treinado, enviamos os dados de teste para ele classificar/prever e avaliarmos

In [ ]:
# testando o modelo NB
y_pred_nb = nb_modelo.predict(X_test_tfidf)

**Resultado**

In [ ]:
print("\nRESULTADOS DO NAIVE BAYES")
print(f"Acurácia: {accuracy_score(y_test, y_pred_nb):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred_nb, target_names=['Verdadeiro (0)', 'Falso (1)']))


RESULTADOS DO NAIVE BAYES
Acurácia: 0.9066

Relatório de Classificação:
                precision    recall  f1-score   support

Verdadeiro (0)       0.86      0.96      0.91       359
     Falso (1)       0.96      0.85      0.90       358

      accuracy                           0.91       717
     macro avg       0.91      0.91      0.91       717
  weighted avg       0.91      0.91      0.91       717



# **Treinando SVM (Support Vector Machine)**

In [ ]:
from sklearn.svm import LinearSVC

#instanciando o modelo SVM
svm_modelo = LinearSVC(random_state = 123)

#treinando
svm_modelo.fit(X_train_tfidf, y_train)

LinearSVC(random_state=123)

Testando...

In [ ]:
# testando o modelo SVM
y_pred_svm = svm_modelo.predict(X_test_tfidf)

**resultado**

In [ ]:
print("\nRESULTADOS DO SVM")
print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred_svm, target_names=['Verdadeiro (0)', 'Falso (1)']))


RESULTADOS DO SVM
Acurácia: 0.9819

Relatório de Classificação:
                precision    recall  f1-score   support

Verdadeiro (0)       0.98      0.98      0.98       359
     Falso (1)       0.98      0.98      0.98       358

      accuracy                           0.98       717
     macro avg       0.98      0.98      0.98       717
  weighted avg       0.98      0.98      0.98       717



# **Treinando CNN (Rede Neural Convolucional)**

Ainda não fiz

# **BASE DE DADOS DA MARIA LUISA**

In [ ]:
df1 = pd.read_csv("/content/FakeRecogna.csv")

In [ ]:
df1.shape[0]

11903

In [ ]:
df1.head(1)

,Titulo,Subtitulo,Noticia,Categoria,Data,Autor,URL,Classe
0,\n\nPapa Francisco foi preso sob acusação de t...,Boato – Ocorreu um apagão no Vaticano. O papa ...,apagão vaticano papar presar acusação tráfico ...,entretenimento,11/01/2021,\nEdgard Matsuki,https://www.boatos.org/religiao/papa-francisco...,0.0


In [ ]:
df1.isna().sum()

,0
Titulo,31
Subtitulo,6323
Noticia,1
Categoria,1
Classe,1


In [ ]:
# RETIRAR COLUNAS
colunas_retirar = ['URL', 'Data', 'Autor']

df1 = df1.drop(columns = colunas_retirar)

In [ ]:
df1.columns

Index(['Titulo', 'Subtitulo', 'Noticia', 'Categoria', 'Classe'], dtype='object')

In [ ]:
#LIMPAR OS TEXTOS
colunas_para_limpar = ['Titulo', 'Subtitulo', 'Noticia']

df1[colunas_para_limpar] = df1[colunas_para_limpar].map(limpar_texto)


Tratando NaN

In [ ]:
#preencher NaN com ""
df1[colunas_para_limpar] = df1[colunas_para_limpar].fillna('')

# eliminar o registro sem classe estabelecida
df1 = df1.dropna(subset=['Classe'])

In [ ]:
df1.head(1)

,Titulo,Subtitulo,Noticia,Categoria,Classe
0,papa francisco preso sob acusação tráfico cria...,boato ocorreu apagão vaticano papa francisco p...,apagão vaticano papar presar acusação tráfico ...,entretenimento,0.0


Juntando as variáveis necessárias em 1 texto só

In [ ]:
df1['texto_completo'] = (
    df1['Titulo'].fillna('') + ' ' +
    df1['Subtitulo'].fillna(' ') + ' ' +
    df1['Noticia'].fillna(' ')
)

/tmp/ipykernel_2850/1987424322.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['texto_completo'] = (


difivindo df1 em teste e treino

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df1['texto_completo'],
    df1['Classe'],
    test_size = 0.2, # 20% de pra teste
    random_state = 123, #sementre da amostra
    stratify = df1['Classe']
)

Vetorizando os dados

In [ ]:
#transformar os dados em matrizes numéricas
vectorizer = TfidfVectorizer(max_features=5000) # mantém as 5k palavras mais importantes

X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

# **Treinando Naive Bayes**



*   X_train_tfidf = dados de treino vetorizados
*   y_train = gabaarito dos dados de treino



In [ ]:
#instanciar o modelo NB
nb_modelo = MultinomialNB()

#treinando o modelo
nb_modelo.fit(X_train_tfidf, y_train)


MultinomialNB()

Teste: com o modelo já treinado, enviamos os dados de teste para ele classificar/prever e avaliarmos

In [ ]:
# testando o modelo NB
y_pred_nb = nb_modelo.predict(X_test_tfidf)

**Resultado**

In [ ]:
print("\nRESULTADOS DO NAIVE BAYES")
print(f"Acurácia: {accuracy_score(y_test, y_pred_nb):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred_nb, target_names=['Verdadeiro (0)', 'Falso (1)']))


RESULTADOS DO NAIVE BAYES
Acurácia: 0.9160

Relatório de Classificação:
                precision    recall  f1-score   support

Verdadeiro (0)       0.91      0.92      0.92      1191
     Falso (1)       0.92      0.91      0.92      1190

      accuracy                           0.92      2381
     macro avg       0.92      0.92      0.92      2381
  weighted avg       0.92      0.92      0.92      2381



support aqui é a proporçao de teste.Neste CASO, 20%

# **Treinando SVM (Support Vector Machine)**

In [ ]:
from sklearn.svm import LinearSVC

#instanciando o modelo SVM
svm_modelo = LinearSVC(random_state = 123)

#treinando
svm_modelo.fit(X_train_tfidf, y_train)

LinearSVC(random_state=123)

testando...

In [ ]:
# testando o modelo SVM
y_pred_svm = svm_modelo.predict(X_test_tfidf)

resultado

In [ ]:

print("\nRESULTADOS DO SVM")
print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred_svm, target_names=['Verdadeiro (0)', 'Falso (1)']))



RESULTADOS DO SVM
Acurácia: 0.9693

Relatório de Classificação:
                precision    recall  f1-score   support

Verdadeiro (0)       0.98      0.96      0.97      1191
     Falso (1)       0.96      0.98      0.97      1190

      accuracy                           0.97      2381
     macro avg       0.97      0.97      0.97      2381
  weighted avg       0.97      0.97      0.97      2381

